# Budowa wielokrotnego użytku biblioteki funkcji wyceny aktuarialnej za pomocą PROC FCMP

## Podsumowanie wykonawcze

Ubezpieczyciel majątkowy zamyka swoją podstawową matematykę wyceny — składkę czystą, narzut kosztów/ryzyka, mieszanie wiarygodności ograniczonej fluktuacji oraz oszacowanie zdyskontowanej rezerwy — w postaci własnych funkcji i wielowyjściowej podprocedury w **PROC FCMP**. Skompilowana biblioteka jest rejestrowana za pomocą opcji systemowej `CMPLIB=`, a następnie wywoływana wiersz po wierszu z kroku DATA, który wycenia syntetyczny portfel 100 polis. Każda wyceniona wartość w tym notatniku — współczynnik wiarygodności `z`, ważona składka czysta, składka pobierana i zdyskontowana rezerwa szkodowa — jest wytwarzana przez te skompilowane procedury, a nie przez arytmetykę wpisaną wprost w kod. Wynik: domniemany wskaźnik szkodowości wynosi 60.5% (wiejski), 55.8% (podmiejski) i 63.6% (miejski) — wyraźnie poniżej 100% w każdym segmencie, co potwierdza, że pobierana składka pokrywa oczekiwaną szkodowość, a sam krok wyceny pozostaje przejrzysty i możliwy do audytu.

## Źródła danych

| Zbiór danych | Wiersze | Opis | Kluczowe zmienne |
|---------|------|-------------|---------------|
| `work.portfolio` | 100 | Syntetyczny, aktywny portfel polis majątkowych wygenerowany wprost za pomocą `rand()` | `policy_id`, `region` (miejski/podmiejski/wiejski), `years_insured`, `exposure` (pojazdolata), `claim_count` (Poisson), `incurred_loss` (dotkliwość gamma x liczba szkód), `class_pure_prem` (stawka ręczna wg regionu) |

Krok DATA iteruje po szerszym zakresie `policy_id`, ale to środowisko działa w trybie nielicencjonowanym, więc wynik jest ograniczony do pierwszych **100 obserwacji** — poniższa wyceniona księga odzwierciedla tych 100 polis.

# Budowa wielokrotnego użytku biblioteki funkcji wyceny aktuarialnej za pomocą PROC FCMP

Aktuariusze powtarzają te same obliczenia w zadaniach wyceny, tworzenia rezerw i raportowania: zamieniają szkody na *składkę czystą*, stosują *narzuty kosztów i ryzyka*, aby uzyskać składkę pobieraną, mieszają własne doświadczenie pojedynczej polisy ze stawką taryfową za pomocą *wiarygodności* i *dyskontują* przyszłe przepływy pieniężne do wartości bieżącej. Przepisywanie tych wzorów w każdym kroku DATA prowadzi do błędów kopiuj-wklej i utrudnia zmiany założeń.

**PROC FCMP** (kompilator funkcji SAS) pozwala zdefiniować każdy wzór raz jako nazwaną funkcję lub podprocedurę, przechować skompilowane procedury w bibliotece, a następnie wywoływać je jak dowolną wbudowaną funkcję SAS. W tym notatniku:

1. Kompilujemy niewielką bibliotekę funkcji aktuarialnych za pomocą `PROC FCMP`.
2. Rejestrujemy ją dla sesji za pomocą opcji systemowej `CMPLIB=`.
3. Generujemy syntetyczny portfel majątkowy.
4. Wyceniamy każdą polisę, wywołując nasze własne funkcje i wielowyjściową podprocedurę z jednego kroku DATA.
5. Podsumowujemy i interpretujemy wyceniony portfel.

## Krok 1 — Wygenerowanie syntetycznego portfela polis

Symulujemy księgę aktywnych polis komunikacyjnych (pierwsze 100 jest wycenianych poniżej pod limitem trybu nielicencjonowanego). Każda polisa należy do regionu taryfowego z własną ręczną *składką czystą* (oczekiwana szkoda na pojazdorok). Liczba szkód podąża za procesem Poissona skalowanym ekspozycją, a dotkliwości podążają za rozkładem gamma, więc `incurred_loss` to realistyczna złożona szkoda (Poisson x gamma). `years_insured` będzie później napędzać wagę wiarygodności. Stałe ziarno losowości poprzez `call streaminit` czyni dane odtwarzalnymi.

In [1]:
DANE portfolio;
    CALL streaminit(20260531);
    DŁUGOŚĆ region $12;
    POWTÓRZ policy_id = 10001 TO 12000;
        /* Przypisanie regionu taryfowego */
        u = rand('uniform');
        JEŚLI u < 0.40 WTEDY POWTÓRZ; region = 'miejski';    base_pp = 820; lambda = 0.18; KONIEC;
        PRZECIWNIE JEŚLI u < 0.75 WTEDY POWTÓRZ; region = 'podmiejski'; base_pp = 540; lambda = 0.11; KONIEC;
        PRZECIWNIE POWTÓRZ; region = 'wiejski';    base_pp = 360; lambda = 0.07; KONIEC;

        /* Staż ubezpieczenia (lata) i ekspozycja (pojazdolata) */
        years_insured = 1 + rand('poisson', 3);
        exposure = round(0.5 + rand('gamma', 4) / 4, 0.01);

        /* Proces złożony szkód: częstość Poissona, dotkliwość gamma */
        claim_count = rand('poisson', lambda * exposure);
        incurred_loss = 0;
        POWTÓRZ c = 1 TO claim_count;
            incurred_loss = incurred_loss + rand('gamma', 2.0) * 2500;
        KONIEC;
        incurred_loss = round(incurred_loss, 0.01);

        /* Ręczna składka czysta taryfowa dla ekspozycji tej polisy */
        class_pure_prem = round(base_pp * exposure, 0.01);

        WYJŚCIE;
    KONIEC;
    ZACHOWAJ policy_id region years_insured exposure claim_count
         incurred_loss class_pure_prem;
WYKONAJ;

PROCEDURA DRUKUJ DANE=portfolio(obs=8) noobs ETYKIETA;
    ETYKIETA policy_id='Nr polisy' region='Region' years_insured='Lata ubezpieczenia'
             exposure='Ekspozycja (pojazdolata)' claim_count='Liczba szkód'
             incurred_loss='Szkoda poniesiona' class_pure_prem='Składka czysta taryfowa';
    TYTUŁ 'Pierwszych 8 symulowanych polis';
WYKONAJ;

                                            Pierwszych 8 symulowanych polis                                             

Nr polisy      Region  Lata ubezpieczenia  Ekspozycja (pojazdolata)   Liczba szkód  Szkoda poniesiona   Składka czysta taryfowa
    10001  wiejski                      5                      1.36              0                  0                     489.6
    10002  miejski                      8                      0.97              1            3432.28                     795.4
    10003  miejski                      2                      1.53              2            7155.92                    1254.6
    10004  podmiejski                   9                       2.4              0                  0                      1296
    10005  wiejski                      5                      0.99              0                  0                     356.4
    10006  podmiejski                   1                      0.83              0                  0         


NOTE: DATA portfolio

NOTE: Unlicensed mode - output limited to 100 observations.

NOTE: Wrote portfolio (100 rows, 7 columns).
NOTE: DATA elapsed:
  wall  0.42 seconds
  cpu   0.42 seconds
NOTE: PROC PRINT data=portfolio

NOTE: PROC PRINT completed: 8 observations printed, 7 variables


## Krok 2 — Skompilowanie biblioteki funkcji aktuarialnych

Teraz sedno notatnika. `PROC FCMP` z `OUTLIB=work.actfuncs.pricing` kompiluje cztery funkcje i jedną podprocedurę do pakietu `pricing` zbioru danych `work.actfuncs`:

- **`pure_premium`** — obserwowana szkoda na jednostkę ekspozycji (połączona częstość x dotkliwość).
- **`credibility_z`** — współczynnik wiarygodności ograniczonej fluktuacji `Z = sqrt(n / (n + k))`, gdzie `n` to lata ekspozycji polisy, a `k` to stała dostrajająca.
- **`charged_premium`** — stosuje proporcjonalny narzut ryzyka i stały wskaźnik kosztów do kosztu szkód, aby uzyskać składkę, którą faktycznie pobieramy.
- **`pv_reserve`** — wartość bieżąca przyszłej wypłaty szkody, `FV / (1+r)**t`, używana do dyskontowania rezerw szkodowych.
- **`blend_premium`** (PODPROCEDURA) — używa `OUTARGS`, aby zwrócić *dwie* wartości naraz: ważoną wiarygodnością składkę czystą oraz użyty współczynnik wiarygodności, dzięki czemu krok DATA otrzymuje obie w jednym wywołaniu.

Każda procedura jest zamykana `ENDSUB`, a podprocedura nazywa swoje zapisywalne argumenty za pomocą `OUTARGS`.

In [2]:
PROCEDURA fcmp outlib=work.actfuncs.pricing;

    /* Obserwowana składka czysta: koszt szkód na jednostkę ekspozycji */
    function pure_premium(loss, exposure);
        JEŚLI exposure <= 0 WTEDY RETURN(.);
        RETURN(loss / exposure);
    endsub;

    /* Wiarygodność ograniczonej fluktuacji: Z = sqrt(n / (n + k)) */
    function credibility_z(n_years, k);
        JEŚLI n_years <= 0 WTEDY RETURN(0);
        RETURN(sqrt(n_years / (n_years + k)));
    endsub;

    /* Składka pobierana = koszt szkód powiększony o narzut ryzyka + koszty */
    function charged_premium(loss_cost, risk_load, expense_ratio);
        gross = loss_cost * (1 + risk_load);
        JEŚLI expense_ratio >= 1 WTEDY RETURN(.);
        RETURN(gross / (1 - expense_ratio));
    endsub;

    /* Wartość bieżąca przyszłej wypłaty szkody zdyskontowana t lat przy stopie r */
    function pv_reserve(future_value, r, t);
        RETURN(future_value / (1 + r) ** t);
    endsub;

    /* Mieszanie wiarygodności: zwraca ważoną składkę czystą ORAZ użyty współczynnik Z */
    subroutine blend_premium(own_pp, class_pp, n_years, k, blended, z_used);
        outargs blended, z_used;
        z_used = credibility_z(n_years, k);
        blended = z_used * own_pp + (1 - z_used) * class_pp;
    endsub;

WYKONAJ;


               The FCMP Procedure

  Output Library: work.actfuncs.pricing
  User-defined Function: pure_premium
  User-defined Function: credibility_z
  User-defined Function: charged_premium
  User-defined Function: pv_reserve
  User-defined Subroutine: blend_premium




NOTE: PROC FCMP outlib=work.actfuncs.pricing

NOTE: Function pure_premium stored to work.actfuncs.pricing.
NOTE: Function credibility_z stored to work.actfuncs.pricing.
NOTE: Function charged_premium stored to work.actfuncs.pricing.
NOTE: Function pv_reserve stored to work.actfuncs.pricing.
NOTE: Subroutine blend_premium stored to work.actfuncs.pricing.


## Krok 3 — Zarejestrowanie biblioteki za pomocą CMPLIB=

Samo skompilowanie procedur nie wystarczy; SAS musi wiedzieć, gdzie ich szukać, gdy późniejszy krok DATA lub procedura odwoła się do nazwy, której nie rozpoznaje jako wbudowanej. Opcja systemowa `CMPLIB=` wskazuje na zbiór danych (nie na trzypoziomową nazwę pakietu) przechowujący skompilowany kod. Po tej instrukcji `OPTIONS` każda funkcja i podprocedura powyżej jest dostępna po nazwie.

In [3]:
OPCJE cmplib=work.actfuncs;


NOTE: Option CMPLIB changed to WORK.ACTFUNCS.


## Krok 4 — Wycena portfela za pomocą własnych funkcji

Krok DATA wyceniający jest teraz niemal wolny od arytmetyki — intencja aktuarialna wynika wprost z nazw funkcji. Dla każdej polisy:

1. Obliczamy własną obserwowaną składkę czystą polisy za pomocą `pure_premium`.
2. Wywołujemy podprocedurę `blend_premium`, aby zważyć ją wiarygodnością względem regionalnej stawki taryfowej, odzyskując zarówno ważony koszt szkód, jak i użyty współczynnik wiarygodności `z` poprzez `OUTARGS`.
3. Podnosimy ważony koszt szkód do składki pobieranej z 12% narzutem ryzyka i 25% wskaźnikiem kosztów za pomocą `charged_premium`.
4. Szacujemy wciąż otwartą rezerwę szkodową jako 35% szkody poniesionej polisy i dyskontujemy ją trzy lata przy stopie 4% do wartości bieżącej za pomocą `pv_reserve`.

Zauważ, że argumenty wyjściowe podprocedury (`blended_pp`, `z`) muszą być zainicjalizowane przed `CALL`. Wartość bieżąca rezerwy różni się polisa od polisy, ponieważ jest napędzana rzeczywistą szkodą poniesioną każdej polisy — polisy bezszkodowe niosą zerową rezerwę, więc ich `reserve_pv` również wynosi zero.

In [4]:
DANE rated;
    USTAW portfolio;

    /* 1. Własne doświadczenie szkodowe polisy jako składka czysta */
    own_pp = round(pure_premium(incurred_loss, exposure), 0.01);

    /* 2. Ważenie wiarygodnością własnego doświadczenia względem stawki taryfowej.
          k = 4 lata ekspozycji dla niemal pełnej wiarygodności. */
    blended_pp = .;
    z = .;
    CALL blend_premium(own_pp, class_pure_prem / exposure,
                       years_insured, 4, blended_pp, z);
    blended_pp = round(blended_pp, 0.01);
    z = round(z, 0.001);

    /* 3. Przeliczenie ważonego kosztu szkód (na pojazdorok) na składkę pobieraną */
    loss_cost = blended_pp * exposure;
    premium = round(charged_premium(loss_cost, 0.12, 0.25), 0.01);

    /* 4. Otwarta rezerwa szkodowa = 35% szkody poniesionej, oczekiwana do
          rozliczenia w 3 lata; dyskontowana do wartości bieżącej przy 4%. */
    case_reserve = round(0.35 * incurred_loss, 0.01);
    reserve_pv   = round(pv_reserve(case_reserve, 0.04, 3), 0.01);

    ZACHOWAJ policy_id region years_insured exposure incurred_loss
         own_pp class_pure_prem blended_pp z premium
         case_reserve reserve_pv;
WYKONAJ;

PROCEDURA DRUKUJ DANE=rated(obs=10) noobs ETYKIETA;
    TYTUŁ 'Wyceniony portfel (pierwsze 10 polis)';
    ZMIENNA policy_id region years_insured exposure own_pp
        blended_pp z premium reserve_pv;
    ETYKIETA policy_id='Nr polisy' region='Region' years_insured='Lata ubezpieczenia'
             exposure='Ekspozycja (pojazdolata)' own_pp='Własna składka czysta'
             blended_pp='Ważona składka czysta' z='Współczynnik wiarygodności (z)'
             premium='Składka' reserve_pv='Wartość bieżąca rezerwy';
WYKONAJ;

                                         Wyceniony portfel (pierwsze 10 polis)                                          

Nr polisy      Region  Lata ubezpieczenia  Ekspozycja (pojazdolata)    Własna składka czysta    Ważona składka czysta     Współczynnik wiarygodności (z)   Składka      Wartość bieżąca rezerwy
    10001  wiejski                      5                      1.36                        0                    91.67                              0.745    186.18                            0
    10002  miejski                      8                      0.97                  3538.43                  3039.59                              0.816   4402.95                      1067.95
    10003  miejski                      2                      1.53                  4677.07                  3046.88                              0.577   6961.51                      2226.55
    10004  podmiejski                   9                       2.4                        0                  


NOTE: DATA rated


NOTE: Read 100 rows from portfolio.
NOTE: Wrote rated (100 rows, 12 columns).
NOTE: DATA elapsed:
  wall  0.06 seconds
  cpu   0.06 seconds
NOTE: PROC PRINT data=rated

NOTE: PROC PRINT completed: 10 observations printed, 9 variables


## Krok 5 — Podsumowanie wycenionej księgi

Ponieważ każda polisa jest wyceniana przez tę samą skompilowaną bibliotekę, możemy zagregować portfel według regionu. Raportujemy średnią składkę pobieraną, średni współczynnik wiarygodności, łączną szkodę poniesioną oraz łączną zdyskontowaną rezerwę szkodową, a następnie wyprowadzamy domniemany wskaźnik szkodowości dla każdego segmentu, aby zobaczyć, czy pobierana składka pokrywa oczekiwaną szkodowość.

In [5]:
PROCEDURA ŚREDNIE DANE=rated mean sum maxdec=2 nonobs;
    KLASA region;
    ZMIENNA premium z incurred_loss reserve_pv;
    ETYKIETA region='Region' premium='Składka' z='Współczynnik wiarygodności (z)'
             incurred_loss='Szkoda poniesiona' reserve_pv='Wartość bieżąca rezerwy';
    TYTUŁ 'Podsumowanie portfela według regionu taryfowego';
WYKONAJ;

/* Domniemany wskaźnik szkodowości = szkoda poniesiona / składka pobierana, plus
   wartość bieżąca rezerwy utrzymywanej przez segment, według regionu. */
PROCEDURA SQL;
    TYTUŁ 'Domniemany wskaźnik szkodowości i wartość bieżąca rezerw wg regionu';
    WYBIERZ region label="Region",
           count(*)                              AS n_policies       label="Liczba polis",
           sum(incurred_loss)                    AS total_loss       format=dollar12.2 label="Szkoda poniesiona łącznie",
           sum(premium)                          AS total_premium    format=dollar12.2 label="Składka łącznie",
           sum(incurred_loss) / sum(premium)     AS loss_ratio       format=percent8.1 label="Wskaźnik szkodowości",
           sum(reserve_pv)                       AS total_pv_reserve format=dollar12.2 label="Rezerwa (wartość bieżąca) łącznie"
    FROM rated
    GROUP WEDŁUG region
    ORDER WEDŁUG region;
QUIT;
TYTUŁ;

                                    Podsumowanie portfela według regionu taryfowego                                     

                                                  The MEANS Procedure

                                          Analysis Variable : premium Składka

        Region               Mean            Sum
        ----------------------------------------
        miejski           1987.17       67563.93
        podmiejski         813.04       34147.74
        wiejski            476.61       11438.62
        ----------------------------------------

                                Analysis Variable : z Współczynnik wiarygodności (z)

        Region               Mean            Sum
        ----------------------------------------
        miejski              0.70          23.90
        podmiejski           0.68          28.36
        wiejski              0.71          17.14
        ----------------------------------------

                                  Analysis Variable 


NOTE: PROC MEANS
NOTE: PROC MEANS statement used.
NOTE: PROC SQL 

NOTE: PROC SQL statement used.


## Interpretacja wyników

Krok DATA wyceniający nigdy nie zapisuje pojedynczego wzoru dyskontowego czy wiarygodności wprost w kodzie — po prostu wywołuje `pure_premium`, `blend_premium`, `charged_premium` i `pv_reserve`. To właśnie korzyść z **PROC FCMP**: założenia aktuarialne żyją w jednej skompilowanej bibliotece, którą można testować jednostkowo, wersjonować i ponownie wykorzystywać w zadaniach wyceny, tworzenia rezerw i raportowania. Zmiana stałej wiarygodności `k`, narzutu ryzyka czy wskaźnika kosztów to pojedyncza edycja w bibliotece, a nie poszukiwanie w każdym programie.

Odczytując wycenioną księgę i zestawienie regionalne:

- **Wiarygodność (`z`)** rośnie wraz z `years_insured`, dokładnie tak, jak dyktuje wzór ograniczonej fluktuacji `Z = sqrt(n / (n + k))`. Wśród pierwszych dziesięciu polis, jednoroczna polisa podmiejska (10006) uzyskuje `z = 0.447`, dwuletnia polisa miejska (10003) `z = 0.577`, podczas gdy dziewięcioletnia polisa podmiejska (10004) osiąga `z = 0.832`. Polisy o skąpym doświadczeniu są ciągnięte w stronę regionalnej stawki taryfowej; te o długim stażu opierają się na własnych szkodach.
- **Mieszanie w akcji:** polisy bezszkodowe (większość księgi) mają `own_pp = 0`, więc `blend_premium` zwraca `blended_pp` bliskie `(1 - z)` razy stawka taryfowa — np. polisa 10001 (wiejski, `z = 0.745`) osiąga `91.67` wobec stawki taryfowej `360`/pojazdorok. Dwie polisy miejskie, które rzeczywiście poniosły szkody, 10002 i 10003, zamiast tego ciągną swój ważony koszt szkód w górę, w stronę własnego wysokiego doświadczenia (`3039.59` i `3046.88`).
- **Składka pobierana** znajduje się powyżej ważonego kosztu szkód, ponieważ `charged_premium` dodaje 12% narzut ryzyka i powiększa o 25% wskaźnik kosztów, czyli stały mnożnik `1.12 / 0.75 = 1.493` na koszcie szkód. Średnia składka pobierana wynosi `476.61` (wiejski), `813.04` (podmiejski) i `1,987.17` (miejski).
- **Zdyskontowane rezerwy:** `pv_reserve` dyskontuje otwartą rezerwę szkodową każdej polisy (35% szkody poniesionej) o trzy lata przy stopie 4%, czyli współczynnik wartości bieżącej `0.889`. Polisy bezszkodowe niosą `reserve_pv = 0`; dwaj miejscy roszczący wnoszą `1,067.95` i `2,226.55`. Zagregowana księga utrzymuje zdyskontowaną rezerwę w wysokości `$2,151.56` (wiejski), `$5,932.52` (podmiejski) i `$13,364.13` (miejski).
- **Domniemane wskaźniki szkodowości** wynoszą 60.5% (wiejski), 55.8% (podmiejski) i 63.6% (miejski) — wszystkie wyraźnie poniżej 100%, więc pobierana składka pokrywa oczekiwaną szkodowość w każdym segmencie. Segment miejski jest najgorętszy, napędzany dwiema swoimi dużymi symulowanymi szkodami; prawdziwy przegląd stawek sprawdziłby, czy ten sygnał utrzymuje się w kolejnych latach szkodowych, zanim skorygowano by stawkę ręczną.

Podprocedura `blend_premium` demonstruje też idiom `OUTARGS` do zwracania wielu wyników z jednego `CALL` — ważony koszt szkód i współczynnik wiarygodności `z` wracają razem — unikając osobnych wywołań funkcji i utrzymując logikę wyceny per polisa zwartą i możliwą do audytu.